# W05-04 · 서비스와 프로젝트 연결

확장 30분. 브라우저 API 테스트, 자기 자료 적용, 기획서 요구사항으로 연결합니다.

**진행 방식**: 먼저 실행 결과를 예측하고 셀을 실행합니다. 변경 실습은 제공된 값 하나를 바꿉니다. 핵심 셀은 네트워크·API 키 없이 실행됩니다. 모든 정책·질의는 교육용 합성 데이터입니다.

VS Code → 우측 상단 Select Kernel → 이 저장소의 .venv (Python 3.11). 노트북별로 Restart 후 Run All 가능합니다.

In [ ]:
from pathlib import Path
import sys, json, math, statistics
from dataclasses import asdict, replace
from collections import Counter
from time import perf_counter

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course_labs").is_dir()), None)
assert ROOT is not None, "저장소 내부에서 notebook을 열어 주세요."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_labs.retrieval import (
    Document,
    Chunk,
    LexicalIndex,
    load_documents,
    load_cases,
    chunk_document,
    tokenize,
    expand_query,
    eligible_chunks,
    reciprocal_rank_fusion,
    rerank_by_coverage,
    retrieval_metrics,
    retrieve,
    answer_from_evidence,
    evaluate,
)
from IPython.display import display, HTML

docs = load_documents()
cases = load_cases()
print("Python:", sys.version.split()[0], "| 문서:", len(docs), "| 평가 질의:", len(cases))
assert sys.version_info[:2] == (3, 11), "과정 기준 Python 3.11.x 커널 선택"

In [ ]:
def show(rows, columns=None):
    import html

    if not rows:
        return display(HTML("<p>결과 없음</p>"))
    cols = columns or list(rows[0])
    headers = "".join(
        '<th style="padding:8px;text-align:left">' + html.escape(str(c)) + "</th>" for c in cols
    )
    body = "".join(
        "<tr>"
        + "".join(
            '<td style="padding:8px;border-bottom:1px solid #ddd">'
            + html.escape(str(r.get(c, "")))
            + "</td>"
            for c in cols
        )
        + "</tr>"
        for r in rows
    )
    display(
        HTML(
            '<table style="font-size:15px;border-collapse:collapse"><tr>'
            + headers
            + "</tr>"
            + body
            + "</table>"
        )
    )

## 01 서비스 실행 안내

Notebook은 서버를 백그라운드로 만들지 않습니다. 별도 터미널을 사용합니다.

In [ ]:
print(".venv/bin/python -m uvicorn week05.lab.search05.main:app --host 127.0.0.1 --port 8005")
print("브라우저: http://127.0.0.1:8005")

**관찰·변경 과제**

VS Code 터미널에서 실행 후 브라우저에 접속하세요. 노트북 커널과 서버 프로세스를 구분합니다.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 02 인프로세스 API 테스트

서버가 없어도 요청과 응답 형식을 확인합니다.

In [ ]:
from fastapi.testclient import TestClient
from course_labs.webapp import create_app

client = TestClient(create_app(5))
print(client.get("/health").json())
response = client.post("/api/search", json={"query": "상품 반품 기간", "method": "rrf"})
assert response.status_code == 200
print(response.json()["answer"])

**관찰·변경 과제**

인프로세스 테스트가 네트워크·배포·인증 테스트를 대신하지 못하는 이유를 적으세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 03 실제 서버 요청

실행 중인 서버에만 선택적으로 요청합니다.

In [ ]:
ENABLE_SERVER = False
if ENABLE_SERVER:
    import requests

    res = requests.post(
        "http://127.0.0.1:8005/api/search", json={"query": "상품 반품 기간"}, timeout=5
    )
    res.raise_for_status()
    print(res.json()["answer"])
else:
    print("SKIP: 서버 실행 후 True로 변경")

**관찰·변경 과제**

8000의 1주차 서비스와 8005의 5주차 서비스를 혼동하지 않습니다.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 04 입력 검증

실패 요청에는 구체적인 오류를 반환합니다.

In [ ]:
for payload in [
    {"query": ""},
    {"query": "반품", "size": 60, "overlap": 60},
    {"query": "반품", "role": "admin"},
]:
    res = client.post("/api/search", json=payload)
    print(res.status_code)
    assert res.status_code == 422

**관찰·변경 과제**

UI에서 오류가 난 뒤 이전 결과를 최신 결과로 오해하지 않도록 변경안을 제안하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 05 자기 자료 검색

프로젝트의 작은 데이터부터 구성합니다.

In [ ]:
project_docs = [
    Document(
        "P01",
        "합성 글쓰기 평가",
        "서론은 주제의 맥락과 제안 아이디어를 소개합니다. 주장1과 주장2는 각각 근거를 제시합니다.",
    ),
    Document(
        "P02",
        "합성 결론 평가",
        "결론은 아이디어를 다시 제언하고 두 주장을 요약합니다. 본문에 없던 새로운 근거를 추가하지 않습니다.",
    ),
]
r = retrieve("결론 아이디어 제언 주장 요약", docs=project_docs)
print(answer_from_evidence(r)["text"])

**관찰·변경 과제**

본인의 서비스 문서 5개와 기대 질의 10개를 만들되 실제 개인정보는 넣지 않습니다.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 06 글쓰기 평가 구조

구조·근거·유창성·이해도를 분리해 기준을 작성합니다.

In [ ]:
rubric = {
    "structure": ["서론", "본론1", "본론2", "결론"],
    "argument": ["주장1 근거", "주장2 근거"],
    "fluency": ["문장 연결", "명확한 표현"],
    "understanding": ["주제 이해", "주장과 근거 관계"],
}
print(json.dumps(rubric, ensure_ascii=False, indent=2))

**관찰·변경 과제**

단어가 들어있다는 이유만으로 구조나 이해도를 통과시키지 않습니다.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 07 근거 있는 피드백

학생 원문 구간과 평가 기준을 연결합니다.

In [ ]:
example_feedback = {
    "criterion": "주장1 근거",
    "span": "합성 원문: AI 도구를 도입하면 좋다.",
    "finding": "주장을 뒷받침하는 구체적인 근거가 없음",
    "suggestion": "작은 비교 실험 또는 출처가 확인되는 관찰 추가",
    "requires_human_review": True,
}
print(json.dumps(example_feedback, ensure_ascii=False, indent=2))

**관찰·변경 과제**

작성자의 의도를 바꿔 완성문을 대신 쓰는 방식과 개선 피드백을 구분하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 08 이해도 확인

확인 질문으로 글의 의도를 파악하되 자동 고위험 판정을 피합니다.

In [ ]:
questions = [
    "이 근거가 주장1을 뒷받침하는 이유는 무엇인가요?",
    "대안 설명은 무엇인가요?",
    "결론의 제안이 본론의 어느 근거와 연결되나요?",
]
for q in questions:
    print(q)

**관찰·변경 과제**

LLM 점수를 공식 학점 결정의 유일한 근거로 사용하지 않는 설계를 쓰세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 09 인수 조건

Acceptance Criteria: 기능 완료를 확인하는 관찰 가능한 조건입니다.

In [ ]:
acceptance = [
    {"id": "AC01", "condition": "모든 피드백에 원문 구간과 기준 ID 포함"},
    {"id": "AC02", "condition": "근거 불명확 시 판단 보류"},
    {"id": "AC03", "condition": "사용자가 기준과 피드백을 수정·삭제 가능"},
]
show(acceptance)

**관찰·변경 과제**

본인의 핵심 기능 3개에 성공·실패·보류 조건을 각각 적으세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 10 데모와 제출 연결

2Pager, PRD, PPT는 서로 다른 수준의 설명입니다.

In [ ]:
deliverables = {
    "2Pager": "사용자 문제·핵심 제안·검증 계획",
    "PRD": "사용 흐름·요구사항·데이터·평가·위험",
    "PPT": "5–8장 한눈에 보는 요약",
    "Prototype": "주요 화면 3–5개",
    "Demo": "60–120초 성공/실패/보류 시연",
}
print(json.dumps(deliverables, ensure_ascii=False, indent=2))

**관찰·변경 과제**

본인의 실험 보고서와 데모 화면을 연결하는 제출 인덱스를 만드세요. 형식은 자유이며 유료 도구 사용은 필수가 아닙니다.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 완료 체크

- 설정 하나를 바꾼 비교 결과와 실패 질의 1개
- 출처·문서 버전·기준 날짜를 포함한 설명
- 측정한 것과 측정하지 않은 것의 구분

## 참고

- [RAG 원 논문](https://arxiv.org/abs/2005.11401)
- [Stanford IR: TF-IDF](https://nlp.stanford.edu/IR-book/html/htmledition/term-frequency-and-weighting-1.html)
- [Stanford IR: BM25](https://nlp.stanford.edu/IR-book/html/htmledition/okapi-bm25-a-non-binary-model-1.html)
- [Ollama embed API](https://docs.ollama.com/api/embed)

로컬 구현 위치: course_labs/retrieval.py. 형태소 분석·학습된 임베딩·신경망 재정렬을 구현한 것으로 해석하지 않습니다.